# Importation et chargement des donnees

In [ ]:
# Importation des librairies
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

import warnings
warnings.filterwarnings('ignore')


# Chargements des donnees
df = pd.read_excel('dataset_pretraitement_fraude.xlsx')

# Supprimer les doublons 
df = df.drop_duplicates(subset=['TransactionID'])

# Visualisation des donnees
print(f"Le dataset a : {df.shape[0]} lignes x {df.shape[1]} colonnes")

# Les informations sur le dataset
print(df.columns)

print("\nApercu sur les donnees : ")
display(df.head(5))

print("\n\nInformations sur les donnees : ")
display(df.info())

# Analyse sur les donnees
print(f"Le nombre de valeur manquantes : {df.isnull().sum()}")

# Les fonctions pour les traitements et nettoyages

In [ ]:
# Fonction pour la numerisation des montants
def montant_clean(montant_raw):
    """
        Convertir le montant en float:
        Les montants du types:
            3, 4500, 05 HGT en 34500.05
            34500.3k en 34500.3*1000
    """
    if pd.isna(montant_raw) or str(montant_raw).strip().lower() in ['n/a', '', 'nan']:
        return np.nan

    # Convertir en str et supprime les espaces vides
    montant_raw = str(montant_raw).strip()

    if 'k' in montant_raw.lower():
        montant_raw = montant_raw.lower().replace('k', '')

        try:
            montant_raw = float(montant_raw.replace(',', ''))*1000
        except:
            return np.nan

    elif 'usd' in montant_raw.lower():
        montant_raw = montant_raw.lower().replace('usd', '')
        try:
            montant_raw = float(montant_raw.replace(',', ''))*130
        except:
            return np.nan
    else:
        montant_raw = montant_raw.lower().replace('htg', '')

        try:
            montant_raw = float(montant_raw.replace(',', ''))
        except:
            return np.nan
    
    return montant_raw

def transform_montant(X):
    """Convertit montant en Gdes.
    - Si contient 'US' => valeur USD * 130
    - Sinon si contient k => valeur milier*1000
    Sinon => valeur deja en Gdes
    - Valeurs non convertibles => NaN
    """
    X = pd.DataFrame(X).copy()
    col = X.columns[0]
    s = X[col]

    # Transformer en string, strip + uppercase
    s_str = s.astype("string").str.strip()
    s_up = s_str.str.upper()

    # Detecter USD
    is_usd = s_up.str.contains("US", na=False)

    # Detecter k (miliers)
    is_k = s_up.str.contains("K", na=False)

    # Retirer "US", ',', 'K' sans regex, puis nettoyer les espaces
    s_clean = (
        s_up.str.replace("US", "", regex=False)
        .str.replace("K", "", regex=False)
        .str.replace("HTG", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )

    # Convertir en float
    num = pd.to_numeric(s_clean, errors="coerce")

    # Conversion: USD -> Gdes * 130, sinon garder tel quel
    out = num.where(~is_usd, num * 130)
    out = out.where(~is_k, out * 1000)

    return pd.DataFrame(out, columns=[col])

# Fonction pour le nettoyage des dates
def date_transaction_clean(date_str):
    """
    Transforme une chaîne de str en objet datetime Python
    """
    # Gestion des valeurs manquantes ou inconnues
    if pd.isna(date_str) or str(date_str).strip().lower() in ['inconnu', 'n/a', '', 'nan']:
        return pd.NaT # Not a Time

    return pd.to_datetime(date_str, errors='coerce')

# Fonction pour le nettoyage de la devise
def devise_clean(devise_raw):
    """
    Nettoyer la colonne devise en supprimant les espaces vides et en mettant en majuscule
    """
    if pd.isna(devise_raw):
        return np.nan

    # Convertir en str, supprimer les espaces vides et mettre en majuscule puis remplacer USD par HTG
    return str(devise_raw).strip().upper().replace('USD', 'HTG')

# Fonction pour convertir l'anciennete en jours
def convertir_anciennete_en_jours(anciennete_str):
    """
    Convertir l'anciennete en jours
    """

    if pd.isna(anciennete_str):
        return np.nan
    
    
    s = str(anciennete_str).strip().lower()
    if s in ['n/a', '', 'nan', 'unknown', 'inconnu']:
        return np.nan

    # Extraire les chiffres
    chiffres = "".join([c for c in s if c.isdigit()])
    
    try:
        nombre = int(chiffres)
    except ValueError:
        return np.nan

    # faire la conversion selon l'unite
    if 'year' in s:
        return nombre * 365
    
    elif 'm' in s:
        return nombre * 30
    
    elif 'day' in s:
        return nombre * 1
    
    else:
        # Si c'est un nombre seul
        return nombre


# # Fonction pour normaliser les villes
def normaliser_ville(ville):
    """
    Normaliser les noms de villes
    """
    if pd.isna(ville) or str(ville).strip().lower() in ['nan', 'inconnu', '']:
        return 'Inconnu'
    
    # tout en minuscule pour comparer
    ville_clean = str(ville).strip().lower()
    
    # Les cles ici DOIVENT etre en minuscules
    normalisation = {
        'p-au-p': 'Port-au-Prince',
        'port au prince': 'Port-au-Prince',
        'pap': 'Port-au-Prince',
        'port-au-prince': 'Port-au-Prince',
        'jacmel': 'Jacmel',
        'cap': 'Cap-Haïtien',
        'cap haitien': 'Cap-Haïtien',
        'cap-haïtien': 'Cap-Haïtien',
        'hin': 'Hinche',
        'hinche': 'Hinche',
        'gonaives': 'Gonaïves',
        'gonaïves': 'Gonaïves',
        'cayes': 'Les Cayes',
        'les cayes': 'Les Cayes'
    }
    
    # renvoie la ville avec la premiere lettre en majuscule
    return normalisation.get(ville_clean, ville.strip().title())

def nettoyer_age(age) -> int:
    """
    Nettoyer la colonne age en gerant les valeurs manquantes et les valeurs aberrantes
    """
    if pd.isna(age) or str(age).strip().lower() in ['n/a', '', 'nan']:
        return np.nan
    
    try:
        age = int(age)
        if age < 0 or age > 100:
            return np.nan
        return age
    except:
        return np.nan

# Fonction pour le nettoyage du revenu
def nettoyer_revenu_or_dette(revenu_or_dette):
    """
    Nettoyer la colonne revenu en gerant les valeurs manquantes
    """
    if pd.isna(revenu_or_dette) or str(revenu_or_dette).strip().lower() in ['n/a', '', 'nan']:
        return np.nan
    
    revenu_or_dette = str(revenu_or_dette).strip()
    try:
        
        val = revenu_or_dette.replace(',', '.') 
        
        revenu = float(val)
        return revenu if revenu >= 0 else np.nan
    except:
        return np.nan

# Fonction pour la normalisation de Employe
def normaliser_employe(employe):
    """
    Normaliser la colonne Employe en convertissant les valeurs 'Yes' et 'No' en 'Oui' et 'Non' respectivement,
    et en gerant les valeurs manquantes
    """
    if pd.isna(employe) or str(employe).strip().lower() in ['n/a', '', 'nan']:
        return 'Inconnu'
    employe = str(employe).strip().title()

    return str(employe).strip().title().replace('No', 'Non').replace('Yes', 'Oui').replace('Nonn', 'Non')

# Fonction pour traiter les Device
def normaliser_other(device):
    """
    Normaliser la colonne Device en gerant les valeurs manquantes
    """
    if pd.isna(device) or str(device).strip().lower() in ['n/a', '', 'nan']:
        return 'Inconnu'
    
    return str(device).strip().title()

# Fonction pour le nettoyage complet
def nettoyer_donnees(data):
    """
    Appliquer toutes les fonctions de nettoyage sur le DataFrame
    """
    # Appliquer les fonctions de nettoyage aux colonnes correspondantes
    # La colonne TODO_Montant_HTG contiendra les montants nettoyes en HTG
    data['TODO_Montant_HTG'] = data['Montant_raw'].apply(montant_clean)

    # Colonne TODO_DateTransaction contiendra les dates nettoyees
    data['TODO_DateTransaction'] = data['DateTransaction_raw'].apply(date_transaction_clean)

    # Colonne TODO_Age_clean contiendra les ages nettoyees
    data['TODO_Age_clean'] = data['Age'].apply(nettoyer_age)

    # Nettoyage de la devise
    data['Devise_indiquee'] = data['Devise_indiquee'].apply(devise_clean)

    # Nettoyage du revenu mensuel
    data['TODO_Revenu_clean'] = data['RevenuMensuel_raw'].apply(nettoyer_revenu_or_dette)

    # Nettoyage de la dette
    data['TODO_Dette_clean'] = data['Dette_raw'].apply(nettoyer_revenu_or_dette)

    # Conversion de l'anciennete en jours
    data['TODO_Anciennete_jours'] = data['AncienneteCompte_raw'].apply(convertir_anciennete_en_jours)

    # Extraction du jour de la semaine en chiffre
    data['TODO_JourSemaine'] = data['TODO_DateTransaction'].dt.dayofweek + 1

    # Extraction de la date du mois
    data['TODO_JourMois'] = data['TODO_DateTransaction'].dt.day

    # Extraction du mois et de l'annee en chiffre
    data['TODO_Mois'] = data['TODO_DateTransaction'].dt.month

    # Extraction de l'annee
    data['TODO_Annee'] = data['TODO_DateTransaction'].dt.year

    # Normalisation des villes
    data['TODO_Ville_norm'] = data['Ville_raw'].apply(normaliser_ville)

    # Gerer les valeurs aberrantes
    data['TODO_Age_clean'] = data['TODO_Age_clean'].clip(lower=18, upper=100)
    data['NbTrans_24h'] = data['NbTrans_24h'].clip(lower=1, upper=24)

    # Normalisation de la colonne Employe
    data['Employe'] = data['Employe'].apply(normaliser_employe)
    
    # Normalisation de la colonne Device
    data['Device'] = data['Device'].apply(normaliser_other)
    data['NiveauEtude'] = data['NiveauEtude'].apply(normaliser_other)
    data['StatutMarital'] = data['StatutMarital'].apply(normaliser_other)

    return data

# Appliquer les fonctions aux donnees pour les nettoyages

## Supprimer les colonnes non necessaires

In [ ]:
# Creation d'une copie des donnees
data = df.copy().drop(columns=['TransactionID', 'ClientID', 'Commentaire'])

# Nettoyage complet des donnees
data = nettoyer_donnees(data)

# Enlever les caracteristiques
data = data.drop(
    columns=[
        'Montant_raw',
        'DateTransaction_raw',
        'RevenuMensuel_raw',
        'Dette_raw',
        'AncienneteCompte_raw',
        'Ville_raw',
        'Age',
        'TODO_DateTransaction'
    ]
)

# Analyse sur les donnees
print(f"Le nombre de valeur manquantes :")
display(data.isnull().sum())
print("\nApercu sur les donnees apres nettoyage : ")
display(data.head())

In [33]:
# Separer les caracteristik numeriques et categoriels
X_numeriques = [
    'TODO_Montant_HTG',
    'TODO_Revenu_clean',
    'TODO_Dette_clean',
    'TODO_Anciennete_jours',
    'TODO_JourSemaine',
    'TODO_JourMois',
    'TODO_Mois',
    'TODO_Annee',
    'TODO_Age_clean',
    'NbTrans_24h',
]

X_categoriels = [
    'Devise_indiquee',
    'TODO_Ville_norm',
    'Employe',
    'Device',
    'NiveauEtude',
    'StatutMarital',
]

# Kole yo
X = data[X_numeriques + X_categoriels]

# La colonne cible
y = data['Fraude']

# Separation des donnees
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

display(y_train.head())

145    0
9      0
382    0
538    0
189    0
Name: Fraude, dtype: int64